In [ ]:
#!/usr/bin/env python3
"""
Differential Camouflage Attack on MFA + Mobile-ViT (PyTorch)

Pipeline:
    rawSAR (complex) -> MFA (torch, autograd) -> jet RGB -> Mobile-ViT -> image
                                      ↑
                               attack via D*A

Requirements
------------
- Python 3.10+
- torch, torchvision
- numpy, scipy
- matplotlib (optional, for saving debug images)
- A module `hffh_vit.py` in the same folder that defines the HFFH_ViT network
  exactly as used for training (img_channels=3, patch_size=(8,8)).

Expected files
--------------
Place these in a `data/` directory (or update the paths below):

- data/rawSAR.mat
    variable: adcDataCube   (Nsamp x M x N), complex
- data/D.mat
    variable: D             (Nsamp x (M*N)), complex
- data/desired_attacked_complex_MFA_RMA.mat
    variable: sar_camouflaged  (H x W), complex
- models/hffh_vit_best_epoch_050.pth
    trained Mobile-ViT checkpoint

Usage
-----
    python attack_mfa_mobilevit.py

This will:
  * load the trained Mobile-ViT,
  * load rawSAR, D, and the desired attacked complex image,
  * run a gradient-based optimization over complex gains A (one per aperture),
  * output the attacked Mobile-ViT image and save PNGs for:
        - clean Mobile-ViT output,
        - target image in Mobile-ViT space,
        - attacked output after optimization.
"""

import os
from dataclasses import dataclass

import numpy as np
import scipy.io as sio
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image

from hffh_vit import HFFH_ViT   # you must provide this file (copied from your training notebook)


# ------------------------------------------------------------
# 1. MFA forward model (differentiable, PyTorch)
# ------------------------------------------------------------

@dataclass
class MFAParams:
    F0: float          # start freq (Hz), e.g. 77e9
    c0: float          # lightspeed, e.g. 3e8
    dx: float          # mm
    dy: float          # mm
    z0_mm: float       # target range (mm)
    bbox: tuple        # (xmin, xmax, ymin, ymax) in mm
    FS: float          # sampling rate (Hz)
    K0: float          # chirp slope (Hz/s)
    tI: float          # instrument delay (s)
    nFFTtime: int      # range FFT size, e.g. 1024
    nFFTspace: int     # spatial FFT size, e.g. 1024


def build_matched_filter(params: MFAParams, device, dtype):
    """
    PyTorch version of refMF(params) from MATLAB.

    Returns
    -------
    matched_filter : (nFFTspace, nFFTspace) complex tensor
    """
    nF = params.nFFTspace
    dx_m = params.dx * 1e-3
    dy_m = params.dy * 1e-3
    z0_m = params.z0_mm * 1e-3

    # index grid like MATLAB: (-(nF-1)/2 : (nF-1)/2) * dx
    idx = torch.arange(nF, device=device, dtype=dtype) - (nF / 2 - 0.5)
    x = dx_m * idx                         # (nF,)
    y = dy_m * idx.view(-1, 1)             # (nF,1) -> broadcast

    k = 2.0 * torch.pi * params.F0 / params.c0
    R = torch.sqrt(x**2 + y**2 + z0_m**2)  # (nF, nF)

    phase = -1j * 2.0 * k * R              # -j 2 k R
    matched_filter = torch.exp(phase)
    return matched_filter


def mfa_image_from_slice(sarData: torch.Tensor, params: MFAParams):
    """
    Differentiable PyTorch version of dlMFA.

    Parameters
    ----------
    sarData : (M, N) complex tensor
        Range-compressed 2D slice at chosen range bin (after serpentine).

    Returns
    -------
    img_mag_norm : (B, A) real tensor
        RMS-normalized magnitude image.
    img_complex  : (B, A) complex tensor
        Complex image (before magnitude).
    """
    assert torch.is_complex(sarData), "sarData must be complex"
    device = sarData.device
    dtype = sarData.real.dtype

    mf = build_matched_filter(params, device=device, dtype=dtype)  # (nF, nF)

    yPointM, xPointM = sarData.shape
    yPointF, xPointF = mf.shape  # both nFFTspace

    # ---- zero-pad sarData up to matched filter size ----
    pad_y_pre = max((yPointF - yPointM) // 2, 0)
    pad_y_post = max(yPointF - yPointM - pad_y_pre, 0)
    pad_x_pre = max((xPointF - xPointM) // 2, 0)
    pad_x_post = max(xPointF - xPointM - pad_x_pre, 0)

    if pad_y_pre > 0 or pad_y_post > 0:
        pad_y_top = torch.zeros(pad_y_pre, xPointM, dtype=sarData.dtype, device=device)
        pad_y_bot = torch.zeros(pad_y_post, xPointM, dtype=sarData.dtype, device=device)
        sarData = torch.cat([pad_y_top, sarData, pad_y_bot], dim=0)

    if pad_x_pre > 0 or pad_x_post > 0:
        y_curr = sarData.shape[0]
        pad_x_left = torch.zeros(y_curr, pad_x_pre, dtype=sarData.dtype, device=device)
        pad_x_right = torch.zeros(y_curr, pad_x_post, dtype=sarData.dtype, device=device)
        sarData = torch.cat([pad_x_left, sarData, pad_x_right], dim=1)

    # 2D FFT of data and matched filter
    sar_fft = torch.fft.fft2(sarData)
    mf_fft = torch.fft.fft2(mf)

    img_shifted = torch.fft.ifft2(sar_fft * mf_fft)

    # fftshift (2D)
    def fftshift2d(x):
        h, w = x.shape[-2:]
        return torch.roll(torch.roll(x, shifts=h // 2, dims=-2),
                          shifts=w // 2, dims=-1)

    img = fftshift2d(img_shifted)

    # Crop using bbox like MATLAB
    J, I = img.shape
    bbox = params.bbox
    dx = params.dx
    dy = params.dy

    xij0 = round(bbox[0] / dx - 0.5 + I / 2.0)
    xij1 = round(bbox[1] / dx - 0.5 + I / 2.0)
    ykl0 = round(bbox[2] / dy - 0.5 + J / 2.0)
    ykl1 = round(bbox[3] / dy - 0.5 + J / 2.0)

    x0 = max(int(xij0), 0)
    x1 = min(int(xij1), I - 1)
    y0 = max(int(ykl0), 0)
    y1 = min(int(ykl1), J - 1)

    img_cropped = img[y0:y1 + 1, x0:x1 + 1]  # (B, A) complex
    img_complex = torch.flip(img_cropped, dims=[1])  # fliplr

    mag = torch.abs(img_complex)
    rms = torch.sqrt(torch.mean(mag ** 2) + 1e-12)
    img_mag_norm = mag / rms

    return img_mag_norm, img_complex


# ------------------------------------------------------------
# 2. Differentiable "jet-like" colormap (approximate)
# ------------------------------------------------------------

import torch


def jet_torch(x: torch.Tensor) -> torch.Tensor:
    """
    Approximate 'jet' colormap in pure PyTorch, differentiable w.r.t. x.

    Parameters
    ----------
    x : tensor
        Shape (B,1,H,W) or (1,H,W), assumed in [0,1].

    Returns
    -------
    rgb : tensor
        Shape (B,3,H,W) (or (3,H,W)) in [0,1].
    """
    if x.dim() == 3:
        x = x.unsqueeze(0)  # (1,1,H,W) -> treat as batch size 1

    assert x.dim() == 4 and x.size(1) == 1, "x must be (B,1,H,W)"

    x = x.clamp(0.0, 1.0)

    r = torch.clamp(1.5 - torch.abs(4.0 * x - 3.0), 0.0, 1.0)
    g = torch.clamp(1.5 - torch.abs(4.0 * x - 2.0), 0.0, 1.0)
    b = torch.clamp(1.5 - torch.abs(4.0 * x - 1.0), 0.0, 1.0)

    rgb = torch.cat([r, g, b], dim=1)  # (B,3,H,W)
    return rgb


# ------------------------------------------------------------
# 3. Victim forward: rawSAR -> MFA -> jet RGB -> Mobile-ViT
# ------------------------------------------------------------

def compute_k0_range_bin(params: MFAParams) -> int:
    z0_m = params.z0_mm * 1e-3
    k0 = round(params.K0 / params.FS * (2.0 * z0_m / params.c0 + params.tI) * params.nFFTtime)
    return int(k0)


def victim_forward(raw_cube: torch.Tensor,
                   params: MFAParams,
                   model: torch.nn.Module):
    """
    raw_cube : (Nsamp, M, N) complex tensor
    Returns:
        sr_rgb  : (1,3,256,256) Mobile-ViT output in [0,1]
        mfa_mag : (H_img, W_img) magnitude (RMS-normalized)
        mfa_cpx : (H_img, W_img) complex image
    """
    device = raw_cube.device

    # 1) Range FFT along fast-time (dim=0)
    raw_fft = torch.fft.fft(raw_cube, n=params.nFFTtime, dim=0)

    # 2) Choose range bin matching z0
    k0_range_bin = compute_k0_range_bin(params)
    sar_slice = raw_fft[k0_range_bin, :, :]  # (M,N) complex

    # 3) Serpentine correction (flip every other row)
    sar_slice = sar_slice.clone()
    sar_slice[1::2, :] = torch.flip(sar_slice[1::2, :], dims=[1])

    # 4) MFA imaging
    mfa_mag, mfa_cpx = mfa_image_from_slice(sar_slice, params)

    # 5) Map to [0,1] and resize to 256x256
    mfa_mag = mfa_mag - mfa_mag.min()
    mfa_mag = mfa_mag / (mfa_mag.max() + 1e-8)

    mfa_mag_4d = mfa_mag.unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    mfa_mag_256 = F.interpolate(
        mfa_mag_4d, size=(256, 256), mode="bilinear", align_corners=False
    )

    # 6) Differentiable jet colormap -> (1,3,256,256)
    lr_rgb = jet_torch(mfa_mag_256)

    # 7) Mobile-ViT forward (keep grads for attack)
    model.eval()
    sr_rgb = model(lr_rgb.to(device))  # (1,3,256,256)

    sr_rgb = sr_rgb.clamp(0.0, 1.0)
    return sr_rgb, mfa_mag, mfa_cpx


# ------------------------------------------------------------
# 4. Target construction from desired complex image
# ------------------------------------------------------------

def build_target_tensor(desired_complex: np.ndarray,
                        device: torch.device) -> torch.Tensor:
    """
    desired_complex : (H,W) complex numpy array (MATLAB sar_camouflaged)
    Returns:
        target_rgb : (1,3,256,256) float32 tensor in [0,1]
    """
    mag = np.abs(desired_complex).astype(np.float32)
    mag = mag - mag.min()
    mag /= (mag.max() + 1e-8)

    mag_t = torch.from_numpy(mag).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,H,W)
    mag_256 = F.interpolate(mag_t, size=(256, 256),
                            mode="bilinear", align_corners=False)

    target_rgb = jet_torch(mag_256)  # (1,3,256,256)
    return target_rgb


# ------------------------------------------------------------
# 5. Load data and model
# ------------------------------------------------------------

def load_raw_sar(path: str, device: torch.device):
    mat = sio.loadmat(path)
    if "adcDataCube" not in mat:
        raise KeyError(f"'adcDataCube' not found in {path}")
    cube_np = mat["adcDataCube"]  # (Nsamp,M,N), complex
    cube_t = torch.from_numpy(cube_np).to(torch.complex64).to(device)
    return cube_t


def load_D(path: str, Nsamp: int, Np: int, device: torch.device):
    mat = sio.loadmat(path)
    if "D" not in mat:
        raise KeyError(f"'D' not found in {path}")
    D_np = mat["D"]  # (Nsamp, Np) complex
    if D_np.shape != (Nsamp, Np):
        raise ValueError(f"D shape {D_np.shape} does not match (Nsamp={Nsamp}, Np={Np})")
    D_t = torch.from_numpy(D_np).to(torch.complex64).to(device)
    return D_t


def load_desired_complex(path: str):
    mat = sio.loadmat(path)
    if "sar_camouflaged" in mat:
        return mat["sar_camouflaged"]
    elif "desired_attacked_complex" in mat:
        return mat["desired_attacked_complex"]
    else:
        raise KeyError(f"Neither 'sar_camouflaged' nor 'desired_attacked_complex' found in {path}")


def load_mobile_vit(ckpt_path: str, device: torch.device):
    model = HFFH_ViT(img_channels=3, patch_size=(8, 8), dropout=0.0).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    return model


# ------------------------------------------------------------
# 6. Attack via complex gains A over D (rawSAR + D*A)
# ------------------------------------------------------------

def run_attack(raw_cube: torch.Tensor,
               D_t: torch.Tensor,
               params: MFAParams,
               model: torch.nn.Module,
               target_rgb: torch.Tensor,
               num_iters: int = 200,
               lr: float = 1e-2,
               lambda_L2: float = 1e-4,
               Amax: float = 2.0):
    """
    Optimize complex gains A (one per aperture location) such that:

        raw_adv = raw_cube + D_cube * A

    drives MFA+Mobile-ViT output toward target_rgb.

    raw_cube : (Nsamp,M,N) complex
    D_t      : (Nsamp,Np) complex, Np = M*N
    target_rgb : (1,3,256,256)
    """
    device = raw_cube.device
    Nsamp, M, N = raw_cube.shape
    Np = M * N

    # Flatten victim data for consistency with MATLAB
    raw_flat = raw_cube.reshape(Nsamp, Np)  # (Nsamp,Np)

    # Initialize A (complex gains) as real+imag parts
    A_re = torch.zeros(Np, device=device, requires_grad=True)
    A_im = torch.zeros(Np, device=device, requires_grad=True)

    optimizer = torch.optim.Adam([A_re, A_im], lr=lr)

    target_rgb = target_rgb.to(device)

    best_loss = float("inf")
    best_A_re = None
    best_A_im = None

    for it in range(1, num_iters + 1):
        optimizer.zero_grad()

        A_c = torch.complex(A_re, A_im)        # (Np,)
        A_row = A_c.unsqueeze(0)               # (1,Np)

        # raw_adv_flat: (Nsamp,Np)
        raw_adv_flat = raw_flat + D_t * A_row  # broadcast
        raw_adv_cube = raw_adv_flat.reshape_as(raw_cube)  # (Nsamp,M,N)

        # victim forward (full graph, differentiable)
        sr_adv, _, _ = victim_forward(raw_adv_cube, params, model)
        loss_img = F.mse_loss(sr_adv, target_rgb)

        # L2 regularization on A
        loss_reg = lambda_L2 * (A_re.pow(2).mean() + A_im.pow(2).mean())
        loss = loss_img + loss_reg

        loss.backward()
        optimizer.step()

        # Optional projection to cap |A|
        if Amax is not None:
            with torch.no_grad():
                magA = torch.abs(torch.complex(A_re, A_im))
                over = magA > Amax
                if over.any():
                    scale = Amax / (magA[over] + 1e-12)
                    A_re.data[over] *= scale
                    A_im.data[over] *= scale

        with torch.no_grad():
            mse_now = loss_img.item()
            meanA = torch.mean(torch.abs(torch.complex(A_re, A_im))).item()
            maxA = torch.max(torch.abs(torch.complex(A_re, A_im))).item()

            if loss.item() < best_loss:
                best_loss = loss.item()
                best_A_re = A_re.detach().clone()
                best_A_im = A_im.detach().clone()

        if it == 1 or it % 10 == 0 or it == num_iters:
            print(
                f"Iter {it:03d}/{num_iters} | "
                f"Loss={loss.item():.4e} | ImgMSE={mse_now:.4e} | "
                f"mean|A|={meanA:.3e}, max|A|={maxA:.3e}"
            )

    print(f"Attack optimization done. Best loss = {best_loss:.4e}")
    return best_A_re, best_A_im


# ------------------------------------------------------------
# 7. Utility: save RGB tensors as PNGs
# ------------------------------------------------------------

def save_rgb_tensor(img: torch.Tensor, path: str):
    """
    img : (1,3,H,W) or (3,H,W) tensor in [0,1]
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if img.dim() == 4:
        img = img[0]
    img = img.clamp(0.0, 1.0).cpu().permute(1, 2, 0).numpy()
    img_uint8 = (img * 255.0).astype(np.uint8)
    Image.fromarray(img_uint8).save(path)


# ------------------------------------------------------------
# 8. Main
# ------------------------------------------------------------

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # ---- Paths (edit as needed) ----
    RAW_SAR_PATH = "data/rawSAR.mat"
    D_PATH = "data/D.mat"
    TARGET_COMPLEX_PATH = "data/desired_attacked_complex_MFA_RMA.mat"
    CKPT_PATH = "models/hffh_vit_best_epoch_050.pth"

    # ---- MFA parameters (match MATLAB) ----
    params = MFAParams(
        F0=77e9,
        c0=3e8,
        dx=1.0,
        dy=1.0,
        z0_mm=185.0,
        bbox=(-200.0, 200.0, -200.0, 200.0),
        FS=5_000_000.0,
        K0=70.295e12,
        tI=4.5225e-10,
        nFFTtime=1024,
        nFFTspace=1024,
    )

    # ---- Load model and data ----
    model = load_mobile_vit(CKPT_PATH, device=device)
    print(f"Loaded Mobile-ViT checkpoint from {CKPT_PATH}")

    raw_cube = load_raw_sar(RAW_SAR_PATH, device=device)
    Nsamp, M, N = raw_cube.shape
    Np = M * N
    print(f"rawSAR cube shape: {raw_cube.shape} (Nsamp={Nsamp}, M={M}, N={N})")

    D_t = load_D(D_PATH, Nsamp=Nsamp, Np=Np, device=device)
    print(f"D shape: {D_t.shape} (Nsamp={Nsamp}, Np={Np})")

    desired_complex = load_desired_complex(TARGET_COMPLEX_PATH)
    target_rgb = build_target_tensor(desired_complex, device=device)
    print(f"Built target tensor in Mobile-ViT space: {tuple(target_rgb.shape)}")

    # ---- Compute clean Mobile-ViT output (no attack) ----
    sr_clean, _, _ = victim_forward(raw_cube, params, model)
    save_rgb_tensor(sr_clean, "outputs/clean_mobilevit.png")
    save_rgb_tensor(target_rgb, "outputs/target_mobilevit.png")
    print("Saved clean and target images to outputs/")

    # ---- Run attack optimization on A ----
    best_A_re, best_A_im = run_attack(
        raw_cube=raw_cube,
        D_t=D_t,
        params=params,
        model=model,
        target_rgb=target_rgb,
        num_iters=200,
        lr=1e-2,
        lambda_L2=1e-4,
        Amax=2.0,
    )

    # ---- Reconstruct final attacked image using best A ----
    Nsamp, M, N = raw_cube.shape
    Np = M * N
    raw_flat = raw_cube.reshape(Nsamp, Np)
    A_c = torch.complex(best_A_re.to(device), best_A_im.to(device))
    A_row = A_c.unsqueeze(0)                  # (1,Np)

    raw_adv_flat = raw_flat + D_t * A_row     # (Nsamp,Np)
    raw_adv_cube = raw_adv_flat.reshape_as(raw_cube)

    sr_adv, _, _ = victim_forward(raw_adv_cube, params, model)
    save_rgb_tensor(sr_adv, "outputs/attacked_mobilevit.png")
    print("Saved attacked Mobile-ViT image to outputs/attacked_mobilevit.png")


if __name__ == "__main__":
    main()
